# Epileptic Seizure Detection — Production Pipeline
**Setup:** E: drive dataset (165 GB free) · 16 GB RAM · SMOTE + GroupShuffleSplit · No overfitting

In [ ]:
import os
import sys

# ════════════════════════════════════════════════════════════════════════════
# USER CONFIGURATION — CHANGE THESE
# ════════════════════════════════════════════════════════════════════════════
DATA_DRIVE = 'E'  # Portable hard drive
WORK_DRIVE = 'C'  # Local SSD for cache

# Find your CHB-MIT folder path (e.g., chb-mit-scalp-eeg-database-1.0.0)
data_dir = os.path.join(DATA_DRIVE + ':', 'chb-mit-scalp-eeg-database-1.0.0')
cache_dir = os.path.join(WORK_DRIVE + ':', 'seizure_cache')

# Verify
if not os.path.exists(data_dir):
    print(f"❌ NOT FOUND: {data_dir}")
    print(f"\nExpected structure:")
    print(f"  {data_dir}")
    print(f"    ├── chb01/")
    print(f"    │   ├── chb01_01.edf")
    print(f"    │   ├── chb01-summary.txt")
    print(f"    └── chb02/")
    print(f"\nUpdate 'data_dir' above with your actual path, then re-run.")
    sys.exit(1)

os.makedirs(cache_dir, exist_ok=True)

print(f"✓ Data directory  : {data_dir}")
print(f"✓ Cache directory : {cache_dir}")
print(f"✓ Ready to begin loading...")

In [ ]:
import numpy as np
import pandas as pd
import mne
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, auc, f1_score, accuracy_score,
                             recall_score, precision_score)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import RandomizedSearchCV
from imblearn.over_sampling import SMOTE
from scipy import signal, stats
import joblib
import glob
import re
import gc
import psutil
import warnings
warnings.filterwarnings('ignore')

print(f"✓ TensorFlow  : {tf.__version__}")
print(f"✓ MNE         : {mne.__version__}")

mem = psutil.virtual_memory()
print(f"✓ RAM available: {mem.available / 1024**3:.1f} GB")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════════════════════════════════════
FS          = 256
WINDOW_SEC  = 5
WINDOW_SIZE = FS * WINDOW_SEC
MAX_WINDOWS_NORMAL = 4000

COMMON_CHANNELS = [
    'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
    'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'FZ-CZ',  'CZ-PZ',
    'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2',
]

USE_SMOTE = True
SMOTE_RATIO = 0.3  # Oversample seizures to 30% of normal

print(f"Config:")
print(f"  Window      : {WINDOW_SEC}s ({WINDOW_SIZE} samples)")
print(f"  Channels    : {len(COMMON_CHANNELS)}")
print(f"  SMOTE       : {USE_SMOTE} (ratio={SMOTE_RATIO})")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# PARSE SEIZURES FROM SUMMARY FILES
# ════════════════════════════════════════════════════════════════════════════
def parse_seizure_registry(data_dir):
    seizure_registry = {}
    summary_files = glob.glob(os.path.join(data_dir, '*', '*-summary.txt'))
    
    if not summary_files:
        print(f"⚠ No summary files found")
        return seizure_registry

    for summary_file in sorted(summary_files):
        try:
            with open(summary_file, 'r') as f:
                lines = f.readlines()

            current_file  = None
            pending_start = None

            for line in lines:
                if 'File Name:' in line:
                    m = re.search(r'File Name:\s+(\S+\.edf)', line, re.IGNORECASE)
                    if m:
                        current_file = m.group(1)
                elif 'Seizure' in line and 'Start Time:' in line:
                    m = re.search(r'Start Time:\s+(\d+)\s+seconds', line)
                    if m:
                        pending_start = int(m.group(1))
                elif 'Seizure' in line and 'End Time:' in line:
                    m = re.search(r'End Time:\s+(\d+)\s+seconds', line)
                    if m and pending_start is not None and current_file is not None:
                        seizure_registry.setdefault(current_file, []).append(
                            (pending_start, int(m.group(1)))
                        )
                        pending_start = None
        except Exception as e:
            print(f"  ⚠ {summary_file}: {e}")

    total = sum(len(v) for v in seizure_registry.values())
    print(f"✓ Parsed {len(seizure_registry)} files with {total} seizure events")
    return seizure_registry

seizure_registry = parse_seizure_registry(data_dir)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# EXTRACT WINDOWS (MEMORY-SAFE: preload=False, immediate cleanup)
# ════════════════════════════════════════════════════════════════════════════
def extract_labeled_windows(file_path, is_seizure_file=False, seizure_registry=None):
    try:
        raw = mne.io.read_raw_edf(file_path, preload=False, verbose=False)
        valid_picks = [ch for ch in COMMON_CHANNELS if ch in raw.ch_names]
        if not valid_picks:
            return np.zeros((0, WINDOW_SIZE * len(COMMON_CHANNELS)), np.float32), \
                   np.array([], dtype=np.int8), 0
        
        raw.pick(valid_picks)
        n_ch = len(raw.ch_names)
        raw.filter(l_freq=0.5, h_freq=40.0, method='fir', verbose=False)
        
        n_samples = raw.n_times
        fname = os.path.basename(file_path)
        windows_list = []
        labels_list = []
        
        # Seizure windows
        if is_seizure_file and seizure_registry and fname in seizure_registry:
            for start_s, end_s in seizure_registry[fname]:
                start_idx = int(start_s * FS)
                end_idx = min(int(end_s * FS), n_samples)
                for i in range(start_idx, end_idx - WINDOW_SIZE, WINDOW_SIZE):
                    data, _ = raw[:, i:i+WINDOW_SIZE]
                    windows_list.append(data.flatten().astype(np.float32))
                    labels_list.append(1)
        
        # Normal windows (capped)
        normal_count = 0
        for i in range(0, n_samples - WINDOW_SIZE, WINDOW_SIZE):
            if normal_count >= MAX_WINDOWS_NORMAL:
                break
            data, _ = raw[:, i:i+WINDOW_SIZE]
            windows_list.append(data.flatten().astype(np.float32))
            labels_list.append(0)
            normal_count += 1
        
        del raw
        gc.collect()
        
        if windows_list:
            return np.array(windows_list, dtype=np.float32), \
                   np.array(labels_list, dtype=np.int8), n_ch
        return np.zeros((0, WINDOW_SIZE * n_ch), np.float32), \
               np.array([], dtype=np.int8), n_ch
    
    except Exception as e:
        fname = os.path.basename(file_path)
        print(f"  ✗ {fname}: {str(e)[:50]}")
        return np.zeros((0, WINDOW_SIZE * len(COMMON_CHANNELS)), np.float32), \
               np.array([], dtype=np.int8), 0

print("✓ Window extraction function ready")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# LOAD ALL EDF FILES (streaming approach = safe with 16 GB RAM)
# ════════════════════════════════════════════════════════════════════════════
edf_files = sorted(glob.glob(os.path.join(data_dir, '*', '*.edf')))
print(f"\nFound {len(edf_files)} EDF files")

if len(edf_files) == 0:
    print(f"❌ NO FILES FOUND in {data_dir}\\*\\*.edf")
    sys.exit(1)

print(f"\nLoading and extracting windows...")
all_X = []
all_y = []
file_groups = []
file_idx = 0

for file_path in edf_files:
    fname = os.path.basename(file_path)
    is_seizure = fname in seizure_registry
    
    X, y, n_ch = extract_labeled_windows(file_path, is_seizure_file=is_seizure,
                                          seizure_registry=seizure_registry)
    
    if len(X) > 0:
        all_X.append(X)
        all_y.append(y)
        file_groups.extend([file_idx] * len(y))
        
        seizure_count = int(y.sum())
        normal_count = len(y) - seizure_count
        marker = '🔴' if seizure_count > 0 else '⚪'
        print(f"  {marker} {fname}: {normal_count:4d} normal, {seizure_count:4d} seizure")
        file_idx += 1
    
    if file_idx % 5 == 0:
        mem = psutil.virtual_memory()
        print(f"    ↳ Memory: {mem.percent:.1f}% ({mem.available/1024**3:.1f} GB free)")
    
    del X, y
    gc.collect()

X_raw = np.concatenate(all_X, axis=0)
y = np.concatenate(all_y, axis=0)
groups = np.array(file_groups, dtype=np.int32)

print(f"\n✓ Dataset loaded:")
print(f"  Total windows  : {len(y):,}")
print(f"  Normal windows : {int((y==0).sum()):,}")
print(f"  Seizure windows: {int((y==1).sum()):,}")
print(f"  Imbalance      : 1 : {int((y==0).sum()) / int((y==1).sum()):.1f}")
print(f"  Unique files   : {len(np.unique(groups))}")

del all_X, all_y
gc.collect()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# PATIENT-AWARE TRAIN/TEST SPLIT (GroupShuffleSplit = no data leakage)
# ════════════════════════════════════════════════════════════════════════════
print("\nPatient-aware train/test split (GroupShuffleSplit)...")
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_raw, y, groups=groups))

X_train_raw = X_raw[train_idx]
X_test_raw  = X_raw[test_idx]
y_train     = y[train_idx].astype(int)
y_test      = y[test_idx].astype(int)

print(f"Scaling (RobustScaler on training data only)...")
scaler = RobustScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

joblib.dump(scaler, 'scaler.pkl')

print(f"\n✓ Train/test split (patient-aware):")
print(f"  Train: {len(y_train):,} ({int((y_train==0).sum()):,} normal, {int((y_train==1).sum()):,} seizure)")
print(f"  Test : {len(y_test):,} ({int((y_test==0).sum()):,} normal, {int((y_test==1).sum()):,} seizure)")
print(f"✓ scaler.pkl saved")

del X_raw, X_train_raw, X_test_raw
gc.collect()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# SMOTE: HANDLE CLASS IMBALANCE (on training data ONLY)
# ════════════════════════════════════════════════════════════════════════════
print(f"\nClass imbalance BEFORE SMOTE:")
print(f"  Minority (seizure) : {int((y_train==1).sum()):,}")
print(f"  Majority (normal)  : {int((y_train==0).sum()):,}")
print(f"  Ratio              : 1 : {int((y_train==0).sum()) / int((y_train==1).sum()):.1f}")

if USE_SMOTE:
    print(f"\nApplying SMOTE (sampling_strategy={SMOTE_RATIO})...")
    smote = SMOTE(sampling_strategy=SMOTE_RATIO, random_state=42, n_jobs=-1)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
    
    print(f"\nAfter SMOTE:")
    print(f"  Minority (seizure) : {int((y_train_smote==1).sum()):,}")
    print(f"  Majority (normal)  : {int((y_train_smote==0).sum()):,}")
    print(f"  Ratio              : 1 : {int((y_train_smote==0).sum()) / int((y_train_smote==1).sum()):.1f}")
    
    X_train = X_train_smote
    y_train = y_train_smote
    del X_train_smote, y_train_smote
else:
    print("\nSMOTE disabled")

gc.collect()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# EXTRACT FEATURES FOR RANDOM FOREST
# ════════════════════════════════════════════════════════════════════════════
N_CHANNELS = X_train.shape[1] // WINDOW_SIZE

def extract_features_batch(X_flat, window_size=WINDOW_SIZE, n_ch=N_CHANNELS, sfreq=FS):
    n_windows = X_flat.shape[0]
    X_3d = X_flat.reshape(n_windows, window_size, n_ch)
    features = []

    for i in range(n_windows):
        win_feats = []
        for ch in range(n_ch):
            d = X_3d[i, :, ch].astype(np.float64)

            # Time-domain
            win_feats += [
                d.mean(), d.std(), d.var(),
                np.percentile(d, 25), np.percentile(d, 75),
                float(stats.skew(d)), float(stats.kurtosis(d)),
                d.min(), d.max(),
            ]

            # Band power
            freqs, psd = signal.welch(d, sfreq, nperseg=min(sfreq, window_size))
            win_feats += [
                float(psd[(freqs >= 0.5)  & (freqs < 4)].mean()),   # Delta
                float(psd[(freqs >= 4)    & (freqs < 8)].mean()),   # Theta
                float(psd[(freqs >= 8)    & (freqs < 13)].mean()),  # Alpha
                float(psd[(freqs >= 13)   & (freqs < 30)].mean()),  # Beta
                float(psd[(freqs >= 30)   & (freqs < 50)].mean()),  # Gamma
            ]
        features.append(win_feats)

    return np.array(features, dtype=np.float32)

print(f"Extracting features from {len(X_train):,} training windows...")
X_train_feat = extract_features_batch(X_train)

print(f"Extracting features from {len(X_test):,} test windows...")
X_test_feat = extract_features_batch(X_test)

print(f"✓ Features: {X_train_feat.shape}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CLASS WEIGHTS (for imbalance handling in models)
# ════════════════════════════════════════════════════════════════════════════
class_weights_arr = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = {int(cls): float(w) for cls, w in zip(np.unique(y_train), class_weights_arr)}

print(f"Class weights: {class_weights_dict}")
print(f"  → Seizure weighted {class_weights_dict[1]:.1f}x more than normal")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# RANDOM FOREST: BASELINE
# ════════════════════════════════════════════════════════════════════════════
print("\nTraining Random Forest baseline...")
rf_base = RandomForestClassifier(
    n_estimators=100, max_depth=12, class_weight='balanced',
    random_state=42, n_jobs=-1, verbose=1,
)
rf_base.fit(X_train_feat, y_train)

rf_test_pred = rf_base.predict(X_test_feat)
rf_test_proba = rf_base.predict_proba(X_test_feat)[:, 1]
rf_train_pred = rf_base.predict(X_train_feat)

rf_train_acc = accuracy_score(y_train, rf_train_pred)
rf_test_acc = accuracy_score(y_test, rf_test_pred)
rf_sens = recall_score(y_test, rf_test_pred, zero_division=0)
rf_spec = recall_score(y_test, rf_test_pred, pos_label=0, zero_division=0)
rf_f1 = f1_score(y_test, rf_test_pred, zero_division=0)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_test_proba)
rf_auc = auc(rf_fpr, rf_tpr)

print(f"\n✓ RF Baseline:")
print(f"  Train Acc  : {rf_train_acc:.4f}")
print(f"  Test Acc   : {rf_test_acc:.4f}  (gap: {rf_train_acc-rf_test_acc:.4f})")
print(f"  Sensitivity: {rf_sens:.4f}")
print(f"  Specificity: {rf_spec:.4f}")
print(f"  F1-score   : {rf_f1:.4f}")
print(f"  ROC-AUC    : {rf_auc:.4f}")
print(f"\n{classification_report(y_test, rf_test_pred, target_names=['Normal', 'Seizure'], zero_division=0)}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# RANDOM FOREST: HYPERPARAMETER TUNING
# ════════════════════════════════════════════════════════════════════════════
print("\nTuning Random Forest (RandomizedSearchCV, 5 iters)...")
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    param_dist, n_iter=5, cv=3, scoring='f1', verbose=1, random_state=42, n_jobs=-1,
)
rf_search.fit(X_train_feat, y_train)

best_rf = rf_search.best_estimator_
best_rf_pred = best_rf.predict(X_test_feat)
best_rf_proba = best_rf.predict_proba(X_test_feat)[:, 1]
best_rf_train_pred = best_rf.predict(X_train_feat)

best_rf_train_acc = accuracy_score(y_train, best_rf_train_pred)
best_rf_test_acc = accuracy_score(y_test, best_rf_pred)
best_rf_sens = recall_score(y_test, best_rf_pred, zero_division=0)
best_rf_spec = recall_score(y_test, best_rf_pred, pos_label=0, zero_division=0)
best_rf_f1 = f1_score(y_test, best_rf_pred, zero_division=0)
best_rf_fpr, best_rf_tpr, _ = roc_curve(y_test, best_rf_proba)
best_rf_auc = auc(best_rf_fpr, best_rf_tpr)

print(f"\n✓ Tuned RF:")
print(f"  Params     : {rf_search.best_params_}")
print(f"  Train Acc  : {best_rf_train_acc:.4f}")
print(f"  Test Acc   : {best_rf_test_acc:.4f}  (gap: {best_rf_train_acc-best_rf_test_acc:.4f})")
print(f"  Sensitivity: {best_rf_sens:.4f}")
print(f"  Specificity: {best_rf_spec:.4f}")
print(f"  F1-score   : {best_rf_f1:.4f}")
print(f"  ROC-AUC    : {best_rf_auc:.4f}")
print(f"\n{classification_report(y_test, best_rf_pred, target_names=['Normal', 'Seizure'], zero_division=0)}")

joblib.dump(best_rf, 'best_rf_tuned_model.pkl')
print("\n✅ best_rf_tuned_model.pkl saved")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CNN: RESHAPE & BUILD
# ════════════════════════════════════════════════════════════════════════════
N_CH_CNN = X_train.shape[1] // WINDOW_SIZE

X_train_cnn = X_train.reshape(-1, WINDOW_SIZE, N_CH_CNN)
X_test_cnn = X_test.reshape(-1, WINDOW_SIZE, N_CH_CNN)

print(f"CNN input: {X_train_cnn.shape}")

def build_cnn(input_shape):
    model = models.Sequential([
        layers.Conv1D(16, 3, activation='relu', input_shape=input_shape,
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4), padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Dropout(0.25),

        layers.Conv1D(32, 3, activation='relu',
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4), padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Dropout(0.25),

        layers.Conv1D(64, 3, activation='relu',
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4), padding='same'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),
        layers.Dropout(0.5),

        layers.Dense(64, activation='relu',
                    kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid'),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
                tf.keras.metrics.Recall(name='sensitivity')],
    )
    return model

print("\nBuilding 1D-CNN...")
cnn_model = build_cnn((WINDOW_SIZE, N_CH_CNN))
cnn_model.summary()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CNN: TRAIN (with early stopping = prevent overfitting)
# ════════════════════════════════════════════════════════════════════════════
print("\nTraining CNN...")

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.3, patience=3, min_lr=1e-6, verbose=1),
]

history = cnn_model.fit(
    X_train_cnn, y_train,
    epochs=30, batch_size=32, validation_split=0.2,
    class_weight=class_weights_dict, callbacks=callbacks, verbose=1,
)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CNN: EVALUATE
# ════════════════════════════════════════════════════════════════════════════
cnn_train_proba = cnn_model.predict(X_train_cnn, verbose=0).flatten()
cnn_test_proba = cnn_model.predict(X_test_cnn, verbose=0).flatten()

cnn_train_pred = (cnn_train_proba > 0.5).astype(int)
cnn_test_pred = (cnn_test_proba > 0.5).astype(int)

cnn_train_acc = accuracy_score(y_train, cnn_train_pred)
cnn_test_acc = accuracy_score(y_test, cnn_test_pred)
cnn_sens = recall_score(y_test, cnn_test_pred, zero_division=0)
cnn_spec = recall_score(y_test, cnn_test_pred, pos_label=0, zero_division=0)
cnn_f1 = f1_score(y_test, cnn_test_pred, zero_division=0)
cnn_fpr, cnn_tpr, _ = roc_curve(y_test, cnn_test_proba)
cnn_auc = auc(cnn_fpr, cnn_tpr)

print(f"\n✓ CNN Results:")
print(f"  Train Acc  : {cnn_train_acc:.4f}")
print(f"  Test Acc   : {cnn_test_acc:.4f}  (gap: {cnn_train_acc-cnn_test_acc:.4f})")
print(f"  Sensitivity: {cnn_sens:.4f}")
print(f"  Specificity: {cnn_spec:.4f}")
print(f"  F1-score   : {cnn_f1:.4f}")
print(f"  ROC-AUC    : {cnn_auc:.4f}")
print(f"\n{classification_report(y_test, cnn_test_pred, target_names=['Normal', 'Seizure'], zero_division=0)}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# FINAL COMPARISON & SAVE
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("FINAL MODEL COMPARISON")
print("="*70)

comp = pd.DataFrame({
    'Metric': ['Train Acc', 'Test Acc', 'Overfitting Gap', 'Sensitivity', 'Specificity', 'F1', 'AUC'],
    'Random Forest': [f'{best_rf_train_acc:.4f}', f'{best_rf_test_acc:.4f}',
                     f'{best_rf_train_acc-best_rf_test_acc:.4f}', f'{best_rf_sens:.4f}',
                     f'{best_rf_spec:.4f}', f'{best_rf_f1:.4f}', f'{best_rf_auc:.4f}'],
    '1D-CNN': [f'{cnn_train_acc:.4f}', f'{cnn_test_acc:.4f}',
              f'{cnn_train_acc-cnn_test_acc:.4f}', f'{cnn_sens:.4f}',
              f'{cnn_spec:.4f}', f'{cnn_f1:.4f}', f'{cnn_auc:.4f}'],
})
print(comp.to_string(index=False))

best = 'CNN' if cnn_test_acc > best_rf_test_acc else 'Random Forest'
print(f"\n⭐ BEST MODEL: {best}")

# Save
cnn_model.save('advanced_model.keras')
joblib.dump(best_rf, 'best_rf_tuned_model.pkl')
joblib.dump(history.history, 'training_history.pkl')

metrics = {
    'dataset_info': {
        'total_windows': int(len(y)),
        'normal_samples': int((y == 0).sum()),
        'seizure_samples': int((y == 1).sum()),
        'num_patients': len(np.unique(groups)),
        'smote_used': USE_SMOTE,
    },
    'random_forest': {
        'test_acc': float(best_rf_test_acc), 'sensitivity': float(best_rf_sens),
        'specificity': float(best_rf_spec), 'f1_score': float(best_rf_f1),
        'roc_auc': float(best_rf_auc),
    },
    'cnn_1d': {
        'test_acc': float(cnn_test_acc), 'sensitivity': float(cnn_sens),
        'specificity': float(cnn_spec), 'f1_score': float(cnn_f1),
        'roc_auc': float(cnn_auc),
    },
    'best_model': best,
}
joblib.dump(metrics, 'model_metrics.pkl')

print(f"\n✅ SAVED:")
print(f"  • scaler.pkl")
print(f"  • best_rf_tuned_model.pkl")
print(f"  • advanced_model.keras")
print(f"  • model_metrics.pkl")
print(f"  • training_history.pkl")
print(f"\n🎉 TRAINING COMPLETE! Run: streamlit run app.py")